# Task 7 (B7) — Enhanced Web Augmentation with Generated Hard Negatives (WDC Products)

Supervisor request: to counter the model over-predicting matches, *generate hard negatives*.
For each original product: ask an LLM (teacher) for names of **similar-but-different** products,
retrieve their offers via Tavily, and pair the original offer with a similar-but-different
offer to form a **hard negative** (near-identical surface form, label 0).

Pipeline (WDC only):
1. **Generate** distractor names — LLM lists similar-but-different products for each entity.
2. **Retrieve** — Tavily search for each distractor name.
3. **Extract + verify** — LLM extracts the web offer and confirms it is a NON-match of the
   original (drop if it is actually the same product or irrelevant).
4. **Compose** two web-v2 training sets and compare:
   - `train_aug_web_v2_balanced.txt` — downsample web positives + add hard negatives → ~1:3 added slice
   - `train_aug_web_v2_hardneg.txt` — keep positives, pile on hard negatives (class weighting balances loss)

Reuses the train-only `web_query_entities.csv` (entity-disjoint). Resumable — re-run to continue.

## Configuration

In [1]:
DATASET       = "wdc-products"     # B7 is product-specific (WDC)
N_DISTRACTORS = 3                   # similar-but-different product names to generate per entity
LLM_MODEL     = "claude-sonnet-4-6"
MAX_RESULTS   = 5                   # Tavily results per distractor query
SEARCH_DEPTH  = "basic"
TOP_K_RESULTS = 2                   # web results per distractor to try to extract
MAX_CONTENT_LEN = 800
MAX_QUERY_LEN = 400
SLEEP_BETWEEN = 0.3
MAX_RETRIES   = 8

## Imports & paths

In [2]:
import json
import sys
import time
from pathlib import Path

import anthropic
import pandas as pd
from dotenv import load_dotenv
from tavily import TavilyClient

ROOT = Path("../").resolve()
sys.path.insert(0, str(ROOT))
from src.data_prep.preprocess import serialize_record, WDC_COLS

PROCESSED = ROOT / "data" / "processed" / DATASET
QUERIES_PATH   = PROCESSED / "web_query_entities.csv"      # train-only entities (reused)
DISTRACTORS    = PROCESSED / "hardneg_distractors.jsonl"
TAVILY_HN      = PROCESSED / "hardneg_tavily.jsonl"
HN_LABELED     = PROCESSED / "hardneg_labeled.jsonl"

load_dotenv(ROOT / ".env")
llm = anthropic.Anthropic(max_retries=MAX_RETRIES)   # SDK retries 429/5xx/529
tavily = TavilyClient()                              # reads TAVILY_API_KEY
COLS = WDC_COLS
print(f"Dataset: {DATASET}  |  entities file: {QUERIES_PATH.name}")

Dataset: wdc-products  |  entities file: web_query_entities.csv


## Step 1 — Generate similar-but-different product names (LLM teacher)

In [3]:
def strip_col_val(text):
    import re
    return re.sub(r"COL \S+ VAL\s*", " ", str(text)).strip()

def distractor_prompt(product_text):
    return (
        "You are helping build hard negative examples for product entity matching.\n"
        f"ORIGINAL PRODUCT:\n{strip_col_val(product_text)}\n\n"
        f"List {N_DISTRACTORS} names of SIMILAR BUT DIFFERENT real products that are easy to "
        "confuse with the original — same category/brand family, but a different model, variant, "
        "capacity, size, colour, edition, or bundle (so they are genuinely DIFFERENT products, "
        "NOT the same product).\n"
        'Reply with valid JSON only: {"names": ["name 1", "name 2", ...]}'
    )

def gen_distractors(product_text):
    for attempt in range(MAX_RETRIES):
        try:
            r = llm.messages.create(model=LLM_MODEL, max_tokens=256,
                                    messages=[{"role": "user", "content": distractor_prompt(product_text)}])
            names = json.loads(r.content[0].text.strip()).get("names", [])
            return [str(n).strip() for n in names if str(n).strip()][:N_DISTRACTORS]
        except (json.JSONDecodeError, KeyError):
            if attempt == MAX_RETRIES - 1:
                return []
            time.sleep(1)
        except anthropic.APIError:
            if attempt == MAX_RETRIES - 1:
                return []
            time.sleep(min(60, 2 ** attempt))

entities = pd.read_csv(QUERIES_PATH)
done = set()
if DISTRACTORS.exists():
    done = {(json.loads(l)["id"], json.loads(l)["side"]) for l in open(DISTRACTORS)}
print(f"Entities: {len(entities)}  |  already done: {len(done)}")

with open(DISTRACTORS, "a") as f:
    for i, row in entities.iterrows():
        if (int(row["id"]), row["side"]) in done:
            continue
        names = gen_distractors(row["text"])
        f.write(json.dumps({"id": int(row["id"]), "side": row["side"], "bucket": row["bucket"],
                            "text": row["text"], "distractors": names}) + "\n")
        f.flush()
        if (i + 1) % 25 == 0:
            print(f"  {i+1}/{len(entities)} entities...")
print("Distractor generation complete.")

Entities: 300  |  already done: 0
  25/300 entities...
  50/300 entities...
  75/300 entities...
  100/300 entities...
  125/300 entities...
  150/300 entities...
  175/300 entities...
  200/300 entities...
  225/300 entities...
  250/300 entities...
  275/300 entities...
  300/300 entities...
Distractor generation complete.


## Step 2 — Tavily search for each distractor name

In [4]:
def search_with_retries(query):
    for attempt in range(MAX_RETRIES):
        try:
            return tavily.search(query, search_depth=SEARCH_DEPTH, max_results=MAX_RESULTS)
        except Exception as e:
            if attempt == MAX_RETRIES - 1:
                return None
            time.sleep(2 ** attempt)

recs = [json.loads(l) for l in open(DISTRACTORS)]
done = set()
if TAVILY_HN.exists():
    done = {(json.loads(l)["id"], json.loads(l)["side"], json.loads(l)["distractor"]) for l in open(TAVILY_HN)}

n = 0
with open(TAVILY_HN, "a") as f:
    for rec in recs:
        for name in rec["distractors"]:
            if (rec["id"], rec["side"], name) in done:
                continue
            resp = search_with_retries(str(name)[:MAX_QUERY_LEN])
            results = [] if resp is None else [
                {"title": r.get("title", ""), "url": r.get("url", ""), "content": r.get("content", "")}
                for r in resp.get("results", [])]
            f.write(json.dumps({"id": rec["id"], "side": rec["side"], "bucket": rec["bucket"],
                                "text": rec["text"], "distractor": name, "results": results}) + "\n")
            f.flush()
            n += 1
            if n % 25 == 0:
                print(f"  {n} distractor searches...")
            time.sleep(SLEEP_BETWEEN)
print(f"Done. {n} new distractor searches.")

  25 distractor searches...
  50 distractor searches...
  75 distractor searches...
  100 distractor searches...
  125 distractor searches...
  150 distractor searches...
  175 distractor searches...
  200 distractor searches...
  225 distractor searches...
  250 distractor searches...
  275 distractor searches...
  300 distractor searches...
  325 distractor searches...
  350 distractor searches...
  375 distractor searches...
  400 distractor searches...
  425 distractor searches...
  450 distractor searches...
  475 distractor searches...
  500 distractor searches...
  525 distractor searches...
  550 distractor searches...
  575 distractor searches...
  600 distractor searches...
  625 distractor searches...
  650 distractor searches...
  675 distractor searches...
  700 distractor searches...
  725 distractor searches...
  750 distractor searches...
  775 distractor searches...
Done. 783 new distractor searches.


## Step 3 — Extract the web offer and verify it is a NON-match (hard negative)

In [5]:
schema = ", ".join(COLS)

def verify_prompt(original_text, web_title, web_url, web_content):
    fields = ", ".join(f'\"{c}\": \"...\"' for c in COLS)
    return (
        "You are extracting a product record from a web result and judging whether it is the SAME "
        "product as an original listing (for entity matching).\n\n"
        f"ORIGINAL PRODUCT:\n{strip_col_val(original_text)}\n\n"
        f"WEB RESULT:\nTitle: {web_title}\nURL: {web_url}\nContent: {web_content[:MAX_CONTENT_LEN]}\n\n"
        "Steps:\n"
        f"1. If the web result does not describe a SPECIFIC product whose fields ({schema}) can be "
        'extracted, reply {"relevant": false} and nothing else.\n'
        "2. Otherwise extract the product and decide if it is the EXACT SAME product as the original "
        "(same model, variant, capacity, size, colour, edition — not just same brand/line). "
        "Pay attention to model number, memory/capacity, size, colour, edition, bundle.\n"
        f'Reply with valid JSON only: {{"relevant": true, "label": 0, "record": {{{fields}}}, '
        '"reasoning": "one sentence"}}  (label=1 if same product, 0 if different).'
    )

def extract_verify(original_text, r):
    for attempt in range(MAX_RETRIES):
        try:
            resp = llm.messages.create(model=LLM_MODEL, max_tokens=400,
                messages=[{"role": "user", "content": verify_prompt(original_text, r["title"], r["url"], r["content"])}])
            return json.loads(resp.content[0].text.strip())
        except (json.JSONDecodeError, KeyError):
            if attempt == MAX_RETRIES - 1:
                return None
            time.sleep(1)
        except anthropic.APIError:
            if attempt == MAX_RETRIES - 1:
                return None
            time.sleep(min(60, 2 ** attempt))

tav = [json.loads(l) for l in open(TAVILY_HN)]
done = set()
if HN_LABELED.exists():
    done = {(json.loads(l)["id"], json.loads(l)["side"], json.loads(l)["url"]) for l in open(HN_LABELED)}

n = 0
with open(HN_LABELED, "a") as f:
    for rec in tav:
        for r in rec["results"][:TOP_K_RESULTS]:
            key = (rec["id"], rec["side"], r.get("url", ""))
            if not r.get("url") or key in done:
                continue
            res = extract_verify(rec["text"], r)
            if res is None or not res.get("relevant"):
                continue
            record = res.get("record", {})
            right_text = serialize_record(record, COLS)
            f.write(json.dumps({"id": rec["id"], "side": rec["side"], "bucket": rec["bucket"],
                                "distractor": rec["distractor"], "url": r.get("url", ""),
                                "left_text": rec["text"], "right_text": right_text,
                                "label": int(res.get("label", 0)),
                                "reasoning": res.get("reasoning", ""), "model": LLM_MODEL}) + "\n")
            f.flush()
            n += 1
            if n % 25 == 0:
                print(f"  {n} verified...")
            time.sleep(SLEEP_BETWEEN)

labeled = [json.loads(l) for l in open(HN_LABELED)]
hard_negs = [r for r in labeled if r["label"] == 0]
dropped_same = sum(1 for r in labeled if r["label"] == 1)
print(f"\nVerified {len(labeled)} web offers → {len(hard_negs)} hard negatives (label 0), "
      f"{dropped_same} dropped as accidental same-product matches.")

  25 verified...
  50 verified...
  75 verified...
  100 verified...
  125 verified...
  150 verified...
  175 verified...
  200 verified...
  225 verified...
  250 verified...
  275 verified...
  300 verified...
  325 verified...
  350 verified...
  375 verified...
  400 verified...
  425 verified...
  450 verified...
  475 verified...
  500 verified...
  525 verified...
  550 verified...
  575 verified...
  600 verified...
  625 verified...
  650 verified...
  675 verified...
  700 verified...
  725 verified...
  750 verified...
  775 verified...
  800 verified...
  825 verified...
  850 verified...
  875 verified...
  900 verified...
  925 verified...
  950 verified...
  975 verified...
  1000 verified...
  1025 verified...
  1050 verified...
  1075 verified...
  1100 verified...
  1125 verified...
  1150 verified...
  1175 verified...
  1200 verified...
  1225 verified...
  1250 verified...
  1275 verified...
  1300 verified...
  1325 verified...
  1350 verified...
  1375 verified.

## Step 4 — Compose the two web-v2 training sets

Reuses the existing `web_labeled.jsonl` (Task 5) for the web *positives*. Builds:
- **balanced**: downsample positives so the added slice is ~1:3 positive
- **hard-neg-heavy**: keep all positives, add all hard negatives

In [6]:
import random
random.seed(42)

base = [l for l in open(PROCESSED / "train.txt", encoding="utf-8") if l.strip()]

# existing Task-5 web slice (positives + its few negatives)
web = [json.loads(l) for l in open(PROCESSED / "web_labeled.jsonl")] if (PROCESSED / "web_labeled.jsonl").exists() else []
web = [r for r in web if r.get("relevant")]
web_pos = [(r["left_text"], r["right_text"]) for r in web if r["label"] == 1]
web_neg = [(r["left_text"], r["right_text"]) for r in web if r["label"] == 0]
gen_neg = [(r["left_text"], r["right_text"]) for r in hard_negs]
print(f"web positives={len(web_pos)}  web negatives={len(web_neg)}  generated hard negs={len(gen_neg)}")

def write_set(path, pos_pairs, neg_pairs):
    with open(path, "w", encoding="utf-8") as f:
        for l in base:
            f.write(l if l.endswith("\n") else l + "\n")
        for lt, rt in pos_pairs:
            f.write(f"{lt}\t{rt}\t1\n")
        for lt, rt in neg_pairs:
            f.write(f"{lt}\t{rt}\t0\n")
    added = len(pos_pairs) + len(neg_pairs)
    pct = 100 * len(pos_pairs) / added if added else 0
    print(f"  {path.name}: {len(base)} base + {added} added ({len(pos_pairs)} pos / {len(neg_pairs)} neg, slice {pct:.0f}% pos)")

all_neg = web_neg + gen_neg

# --- balanced: added slice ~1:3 positive → keep len(all_neg)//3 positives ---
target_pos = max(1, len(all_neg) // 3)
bal_pos = web_pos if len(web_pos) <= target_pos else random.sample(web_pos, target_pos)
print("Balanced variant:")
write_set(PROCESSED / "train_aug_web_v2_balanced.txt", bal_pos, all_neg)

# --- hard-neg-heavy: all positives + all negatives (class weighting balances loss) ---
print("Hard-neg-heavy variant:")
write_set(PROCESSED / "train_aug_web_v2_hardneg.txt", web_pos, all_neg)

web positives=649  web negatives=154  generated hard negs=1290
Balanced variant:
  train_aug_web_v2_balanced.txt: 2500 base + 1925 added (481 pos / 1444 neg, slice 25% pos)
Hard-neg-heavy variant:
  train_aug_web_v2_hardneg.txt: 2500 base + 2093 added (649 pos / 1444 neg, slice 31% pos)


## Next — retrain & compare (run in the training notebook / multiseed)

```
for RUN, FILE in [("web_v2_bal_cw", "train_aug_web_v2_balanced.txt"),
                  ("web_v2_hn_cw",  "train_aug_web_v2_hardneg.txt")]:
    train(dataset="wdc-products", class_weight="balanced", run_name=RUN,
          train_file=PROCESSED / FILE)
    evaluate(dataset="wdc-products", run_name=RUN, save_preds=True)
```
Then `significance.py --baseline baseline_cw --treatment web_v2_bal_cw` (and `web_v2_hn_cw`),
and compare to `string_aug_cw` (0.676) and the Task-5 `web_aug_cw` (0.651).